# Homework 3 - Classification

In [1]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv('https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv')

In [5]:
df

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1
...,...,...,...,...,...,...,...,...,...
4995,organic_search,education,employed,asia,45834.0,2,2,0.37,1
4996,NaN,technology,employed,north_america,84831.0,2,5,0.51,0
4997,organic_search,finance,employed,asia,53636.0,4,6,0.75,1
4998,referral,technology,employed,africa,69430.0,4,8,0.79,1


In [ ]:
df.isnull().sum()

In [6]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [7]:
categorical = df.select_dtypes(include=['object', 'string']).columns.tolist()
numerical = df.select_dtypes(exclude=['object', 'string']).columns.tolist()

In [8]:
df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

In [9]:
df

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NA,0.0,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1
...,...,...,...,...,...,...,...,...,...
4995,organic_search,education,employed,asia,45834.0,2,2,0.37,1
4996,NA,technology,employed,north_america,84831.0,2,5,0.51,0
4997,organic_search,finance,employed,asia,53636.0,4,6,0.75,1
4998,referral,technology,employed,africa,69430.0,4,8,0.79,1


In [10]:
df['industry'].mode()[0]

'technology'

In [11]:
from sklearn.model_selection import train_test_split

In [12]:
categorical = df.select_dtypes(include=['object', 'string']).columns.tolist()
numerical = df.select_dtypes(exclude=['object', 'string']).columns.tolist()

df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

In [13]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

In [14]:
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

In [15]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [16]:
features = ['annual_income', 'number_of_courses_viewed',
            'interaction_count', 'lead_score']

corr = df_train[features].corr()

In [17]:
print(corr)

                          annual_income  number_of_courses_viewed  \
annual_income                  1.000000                  0.166168   
number_of_courses_viewed       0.166168                  1.000000   
interaction_count              0.117885                  0.714902   
lead_score                     0.221734                  0.750809   

                          interaction_count  lead_score  
annual_income                      0.117885    0.221734  
number_of_courses_viewed           0.714902    0.750809  
interaction_count                  1.000000    0.916535  
lead_score                         0.916535    1.000000  


In [18]:
pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count'),
]

for a, b in pairs:
    print(a, b, round(corr.loc[a, b], 4))

interaction_count lead_score 0.9165
number_of_courses_viewed lead_score 0.7508
number_of_courses_viewed interaction_count 0.7149
annual_income interaction_count 0.1179


In [20]:
from sklearn.metrics import mutual_info_score

In [21]:
categorical = ['industry', 'location', 'lead_source', 'employment_status']

In [22]:
for c in categorical:
    score = mutual_info_score(df_train[c], y_train)
    print(c, round(score, 2))

industry 0.0
location 0.0
lead_source 0.03
employment_status 0.02


In [23]:
mi = pd.Series(
    {c: round(mutual_info_score(df_train[c], y_train), 2) for c in categorical}
)

mi_sorted = mi.sort_values(ascending=False)
print(mi_sorted)

lead_source          0.03
employment_status    0.02
location             0.00
industry             0.00
dtype: float64


In [24]:
from sklearn.feature_extraction import DictVectorizer

In [25]:
from sklearn.linear_model import LogisticRegression

In [26]:
from sklearn.metrics import accuracy_score

In [27]:
dv = DictVectorizer(sparse=False)

In [28]:
train_dicts = df_train.to_dict(orient='records')
X_train = dv.fit_transform(train_dicts) 

In [29]:
val_dicts = df_val.to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [30]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

In [31]:
model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l

In [42]:
y_pred = model.predict(X_val)

In [38]:
accuracy = accuracy_score(y_val, y_pred)

In [39]:
accuracy

0.645

In [45]:
def train_and_score(features):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
    X_val = dv.transform(df_val[features].to_dict(orient='records'))

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    return accuracy_score(y_val, model.predict(X_val))

In [46]:
results = []
for f in features:
    subset = [x for x in features if x != f]
    acc = train_and_score(subset)
    results.append((f, acc, accuracy - acc))

In [47]:
res = pd.DataFrame(results, columns=['feature', 'acc_without', 'diff'])
res = res.sort_values('diff')
print(res)

                    feature  acc_without   diff
0             annual_income        0.719 -0.074
1  number_of_courses_viewed        0.645  0.000
3                lead_score        0.644  0.001
2         interaction_count        0.597  0.048


In [48]:
dv = DictVectorizer(sparse=False)

In [49]:
X_train = dv.fit_transform(df_train.to_dict(orient='records'))

In [50]:
X_val = dv.transform(df_val.to_dict(orient='records'))

In [51]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    print(C, round(accuracy_score(y_val, y_pred), 3))

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645
